In [ ]:
import sys
import numpy as np
import pandas as pd
import numpy as np 
from sklearn.decomposition import NMF, non_negative_factorization
from sklearn.decomposition._nmf import _beta_divergence
import os

# Notebook uses functions from .py scripts in scripts/ folder
sys.path.append('../scripts/calculate')

from metrics import cosine_sim, get_evar, rss_calc, l2norm_calc

## Nine-fold Bi-cross Validation

This notebook implements Nine-fold Bi-cross Validation, computing reconstruction metrics for each k from 2 to Kmax and saving the results for each k and fold. 
    
The implementation was adapted from Frioux et al. (2023): https://gitlab.inria.fr/cfrioux/enterosignature-paper

### Functions for biCV

In [ ]:
def biCV_sub(X, h = 3):
    """ 
    Take a matrix, shuffle rows and cols and return 9 submatrices for biCV in dict
    """
    # copy matrix
    X_shuf = np.matrix.copy(np.array(X.T))
    # shuffle rows
    np.random.shuffle(X_shuf)
    # shuffle columns
    np.random.shuffle(np.transpose(X_shuf))
    # cut the hxh submatrices
    nfeatures, nsamples = X_shuf.shape
    print(f'functions: {nfeatures}, samples: {nsamples}')
    chunks_feat = nfeatures // h
    remainings_feat = nfeatures % h
    chunks_samp = nsamples // h
    remainings_samp = nsamples % h

    print(f'chunks feat: {chunks_feat}, remainings feat : {remainings_feat }')
    print(f'chunks samp: {chunks_samp}, remainings samp : {remainings_samp }')

    thresholds_feat = [chunks_feat * i for i in range(1,h)]
    thresholds_feat.insert(0,0)
    thresholds_feat.append(nfeatures+1)
    thresholds_samp = [chunks_samp * i for i in range(1,h)]
    thresholds_samp.insert(0,0)
    thresholds_samp.append(nsamples+1)
    print(f'cut originial X in {thresholds_feat} rows')
    print(f'cut originial X in {thresholds_samp} columns')

    # return the 9 matrices
    count_submatrices = h*h
    # starting from top left block -> top right block 
    all_sub_matrices = {}
    i = 0
    row = 0
    col = 0
    while row < h:
        while col < h:
            i += 1
            all_sub_matrices[i] = X_shuf[thresholds_feat[row] : thresholds_feat[row+1], 
                                        thresholds_samp[col] : thresholds_samp[col+1]]
            col += 1
        row += 1
        col = 0
    assert(len(all_sub_matrices) == count_submatrices)

    for i, sub in all_sub_matrices.items():
        print(f'Block {i} with shape {sub.shape}')

    return all_sub_matrices

In [ ]:
def concat_3x3_mx(m_dict, row1, row2, row3):
    """ 
    Concatenate submatrices based on the order given in arguments
    """
    r1 = np.concatenate((m_dict[row1[0]], m_dict[row1[1]], m_dict[row1[2]]), axis=1) 
    r2 = np.concatenate((m_dict[row2[0]], m_dict[row2[1]], m_dict[row2[2]]), axis=1)
    r3 = np.concatenate((m_dict[row3[0]], m_dict[row3[1]], m_dict[row3[2]]), axis=1)
    m = np.concatenate((r1, r2, r3), axis = 0)
    return m

In [ ]:
def biCV_3x3(sm_dict):
    """ 
    Starting from 9 submatrices, rearrange them to return
    9 matrices with original shape used for biCV
    """
    all_mx = {}
    # m1 (all rows) 123 - 456 - 789
    all_mx[1] = concat_3x3_mx(sm_dict, [1,2,3], [4,5,6], [7,8,9])    
    # m2 312 - 645 - 978
    all_mx[2] = concat_3x3_mx(sm_dict, [3,1,2], [6,4,5], [9,7,8]) 
    # m3 231 - 564 - 897
    all_mx[3] = concat_3x3_mx(sm_dict, [2,3,1], [5,6,4], [8,9,7]) 
    # m4 789 - 123 - 456
    all_mx[4] = concat_3x3_mx(sm_dict, [7,8,9], [1,2,3], [4,5,6])  
    # m5 456 - 789 - 123
    all_mx[5] = concat_3x3_mx(sm_dict, [4,5,6], [7,8,9], [1,2,3])  
    # m6 645 - 978 - 312
    all_mx[6] = concat_3x3_mx(sm_dict, [6,4,5], [9,7,8], [3,1,2])  
    # m7 978 - 312 - 645
    all_mx[7] = concat_3x3_mx(sm_dict, [9,7,8], [3,1,2], [6,4,5])  
    # m8 897 - 231 - 564
    all_mx[8] = concat_3x3_mx(sm_dict, [8,9,7], [2,3,1], [5,6,4])  
    # m9 564 - 897 - 231
    all_mx[9] = concat_3x3_mx(sm_dict, [5,6,4], [8,9,7], [2,3,1])
    return all_mx


In [ ]:
def cut_in_four(m, h=3):
    """ 
    Take a matrix and cuts it in 4 for cross validation following a ratio h
    e.g h = 3: the M1 submatrix for validation will have size 1/9 of the matrix
    """
    nfeatures, nsamples = m.shape
    chunks_feat = nfeatures // h
    chunks_samp = nsamples // h
    thresholds_feat = [chunks_feat * i for i in range(1,h)]
    thresholds_samp = [chunks_samp * i for i in range(1,h)]
    m1 = m[0:thresholds_feat[0], 0:thresholds_samp[0]]
    m2 = m[0:thresholds_feat[0], thresholds_samp[0]:nsamples+1]
    m3 = m[thresholds_feat[0]:nfeatures+1, 0:thresholds_samp[0]]
    m4 = m[thresholds_feat[0]:nfeatures+1, thresholds_samp[0]:nsamples+1]
    return m1, m2, m3, m4

In [9]:
def init_output(outdir):
    os.makedirs(outdir, exist_ok=True)

    files = {
    "evar": open(f"{outdir}/evar.tsv", "w", buffering=1),
    "rss": open(f"{outdir}/rss.tsv", "w", buffering=1),
    "l2": open(f"{outdir}/l2norm.tsv", "w", buffering=1),
    "cosine": open(f"{outdir}/cosine.tsv", "w", buffering=1),
    "reco": open(f"{outdir}/reconstruction_error.tsv", "w", buffering=1),
    "log": open(f"{outdir}/process.log", "w", buffering=1)
    }


    header = "fold\trank\trun\tblock\tvalue\n"
    for k in ["evar", "rss", "l2", "cosine", "reco"]:
        files[k].write(header)

    return files

In [ ]:
def run_cv(mx9, ranks, outdir, maxiter = 2000, alpha=0, nruns = 50, h = 3):
    """ 
    Run CV for a set of 9 matrices corresponding  to shuffled versions of the original matrix
    in which each 1/9 fold becomes the validation set. 
    The NMF algorithm is applied to all 9 matrices for each rank k (number of enterosignatures).
    
    For each combination of matrix and k, the NMF algorithm is permorfed nruns times.
    
    For each run, the values of reconstruction error, cosine similarity, L2 norm, RSS, 
    and explained variance are calculated and return in dictionaries.
    """
    files = init_output(outdir)
    log = files["log"]

    res = {i: None for i in range(1,10)}
    rss = {i: None for i in range(1,10)}
    reco_error = {i: None for i in range(1,10)}
    cosine = {i: None for i in range(1,10)}
    l2_norm = {i: None for i in range(1,10)}

    # For each re-arranged matrices
    for i in range(1, 10):
        log.write(f"\n=== Fold {i} ===\n")
        log.flush()

        mx = mx9[i]
        M1, M2, M3, M4 = cut_in_four(mx, h)
        # M1 (or A) - for getting reconctruction error between M1 and predicted based on new samples and new functions
        # M2 (or B) - for predicting W_a (ES in functions) using new functions and same samples
        # M3 (or C) - for predicting H_a (ES in samples) using new samples and same functions
        # M4 (or D) - get H_d (ES in samples) and W_d (ES in functions)
        res[i] = {r: {"A":[], "B":[], "C":[], "D":[]} for r in ranks}
        rss[i] = {r: {"A":[], "B":[], "C":[], "D":[]} for r in ranks}
        reco_error[i] = {r: {"A":[], "B":[], "C":[], "D":[]} for r in ranks}
        cosine[i] = {r: {"A":[], "B":[], "C":[], "D":[]} for r in ranks}
        l2_norm[i] = {r: {"A":[], "B":[], "C":[], "D":[]} for r in ranks}
        
        for rank in ranks:
            log.write(f"Rank = {rank}\n")
            log.flush()
            
            for run in range(1,nruns+1):
                log.write(f"  Run {run}\n")
                log.flush()

                # Step 1
                # NMF for M_d
                model_D = NMF(n_components=rank,
                            init="nndsvdar",
                            verbose=False, 
                            solver="mu", 
                            max_iter=maxiter,
                            random_state = None,
                            l1_ratio = alpha,
                            alpha_W=alpha,
                            alpha_H=alpha,
                            beta_loss = "kullback-leibler")
                
                H_d = np.transpose(model_D.fit_transform(np.transpose(M4)))
                W_d = np.transpose(model_D.components_)
                Md_calc = W_d.dot(H_d)

                evar_D = get_evar(M4, Md_calc)
                reco_D = model_D.reconstruction_err_
                rss_D = rss_calc(M4, Md_calc)
                l2_norm_D = l2norm_calc(M4, Md_calc)
                cos_D = cosine_sim(M4, Md_calc)
                
                res[i][rank]["D"].append(evar_D)
                reco_error[i][rank]["D"].append(reco_D)
                rss[i][rank]["D"].append(rss_D)
                l2_norm[i][rank]["D"].append(l2_norm_D)
                cosine[i][rank]["D"].append(cos_D)

                files["evar"].write(f"{i}\t{rank}\t{run}\tD\t{evar_D}\n")
                files["rss"].write(f"{i}\t{rank}\t{run}\tD\t{rss_D}\n")
                files["l2"].write(f"{i}\t{rank}\t{run}\tD\t{l2_norm_D}\n")
                files["cosine"].write(f"{i}\t{rank}\t{run}\tD\t{cos_D}\n")
                files["reco"].write(f"{i}\t{rank}\t{run}\tD\t{reco_D}\n")

                # Step 2
                # Get W_a using infromation from M_b.
                # H_d and M_b have the same samples but different functions.
                W_a, H_d_t, n_iter = non_negative_factorization(M2, 
                                                    n_components=rank, 
                                                    init='custom',
                                                    verbose=False,
                                                    solver="mu", 
                                                    max_iter=2000,
                                                    random_state=None,
                                                    alpha_W=alpha,
                                                    alpha_H=alpha,
                                                    beta_loss = "kullback-leibler",
                                                    update_H=False, 
                                                    H=H_d)
                Mb_calc = np.dot(W_a, H_d)
                print(f"shape M2: {M2.shape} -- shape W_a: {W_a.shape} -- shape H_d: {H_d.shape}")

                evar_B = get_evar(M2, Mb_calc)
                reco_B = _beta_divergence(np.array(M2), np.array(W_a), np.array(H_d), "kullback-leibler",
                                                    square_root=True)
                rss_B = rss_calc(M2, Mb_calc)
                l2_norm_B = l2norm_calc(M2, Mb_calc)
                cos_B = cosine_sim(M2, Mb_calc)

                res[i][rank]["B"].append(evar_B)
                reco_error[i][rank]["B"].append(reco_B)
                rss[i][rank]["B"].append(rss_B)
                l2_norm[i][rank]["B"].append(l2_norm_B)
                cosine[i][rank]["B"].append(cos_B)
                
                files["evar"].write(f"{i}\t{rank}\t{run}\tB\t{evar_B}\n")
                files["rss"].write(f"{i}\t{rank}\t{run}\tB\t{rss_B}\n")
                files["l2"].write(f"{i}\t{rank}\t{run}\tB\t{l2_norm_B}\n")
                files["cosine"].write(f"{i}\t{rank}\t{run}\tB\t{cos_B}\n")
                files["reco"].write(f"{i}\t{rank}\t{run}\tB\t{reco_B}\n")

                # Step 3
                # Get H_a using infromation from M_c
                # W_d and M_b have the same functions but different samples
                H_a, W_d_t, n_iter = non_negative_factorization(M3.T, 
                                                    n_components=rank, 
                                                    init='custom',
                                                    verbose=False,
                                                    solver="mu", 
                                                    max_iter=2000,
                                                    random_state=None,
                                                    alpha_W=alpha,
                                                    alpha_H=alpha,
                                                    beta_loss = "kullback-leibler",
                                                    update_H=False, 
                                                    H=W_d.T)


                Mc_calc = np.dot(W_d, H_a.T)

                evar_C = get_evar(M3, Mc_calc)
                reco_C = _beta_divergence(
                    np.array(M3),
                    np.array(W_d),
                    np.array(H_a).T,
                    "kullback-leibler",
                    square_root=True
                )
                rss_C = rss_calc(M3, Mc_calc)
                l2_norm_C = l2norm_calc(M3, Mc_calc)
                cos_C = cosine_sim(M3, Mc_calc)

                res[i][rank]["C"].append(evar_C)
                reco_error[i][rank]["C"].append(reco_C)
                rss[i][rank]["C"].append(rss_C)
                l2_norm[i][rank]["C"].append(l2_norm_C)
                cosine[i][rank]["C"].append(cos_C)

                files["evar"].write(f"{i}\t{rank}\t{run}\tC\t{evar_C}\n")
                files["rss"].write(f"{i}\t{rank}\t{run}\tC\t{rss_C}\n")
                files["l2"].write(f"{i}\t{rank}\t{run}\tC\t{l2_norm_C}\n")
                files["cosine"].write(f"{i}\t{rank}\t{run}\tC\t{cos_C}\n")
                files["reco"].write(f"{i}\t{rank}\t{run}\tC\t{reco_C}\n")

                # Step 4
                # Calculate error for M_a
                # Recontruct M1 from infromation about samples from H_a and about functions from W_a
                Ma_calc = np.dot(W_a, H_a.T)

                evar_A = get_evar(M1, Ma_calc)
                reco_A = _beta_divergence(
                    np.array(M1),
                    np.array(W_a),
                    np.array(H_a).T,
                    "kullback-leibler",
                    square_root=True
                )
                rss_A = rss_calc(M1, Ma_calc)
                l2_norm_A = l2norm_calc(M1, Ma_calc)
                cos_A = cosine_sim(M1, Ma_calc)

                res[i][rank]["A"].append(evar_A)
                reco_error[i][rank]["A"].append(reco_A)
                rss[i][rank]["A"].append(rss_A)
                l2_norm[i][rank]["A"].append(l2_norm_A)
                cosine[i][rank]["A"].append(cos_A)

                files["evar"].write(f"{i}\t{rank}\t{run}\tA\t{evar_A}\n")
                files["rss"].write(f"{i}\t{rank}\t{run}\tA\t{rss_A}\n")
                files["l2"].write(f"{i}\t{rank}\t{run}\tA\t{l2_norm_A}\n")
                files["cosine"].write(f"{i}\t{rank}\t{run}\tA\t{cos_A}\n")
                files["reco"].write(f"{i}\t{rank}\t{run}\tA\t{reco_A}\n")
    
    for f in files.values():
        f.close()

    return res, rss, reco_error, cosine, l2_norm

In [ ]:
def shuffle_and_cv(M, outdir, ranks, maxiter = 2000, nruns = 50, alpha = 0, h = 3):
    """ 
    For each of the n runs:
                            1. Shuffle matrix
                            2. Cut in 9 blocks
                            4. Perform CV
    """
    submx = biCV_sub(M)
    all_9_mx = biCV_3x3(submx)
    res_dict, rss_dict, reco_error_dict, cosine_dict, l2norm_dict = run_cv(mx9 = all_9_mx,
                    outdir=outdir,
                    ranks=ranks, 
                    maxiter=maxiter, 
                    nruns=nruns,
                    alpha=alpha,
                    h=h)
    return {'evar':res_dict, 'rss':rss_dict, 'reconstruction_error':reco_error_dict, 'cosine':cosine_dict, 'l2norm':l2norm_dict}

### Run biCV for given range of topics

In [ ]:
TOPICS = [i for i in range(2,21)] # Calculate NMF for k from 2 to 20
X_train = pd.read_csv('../data/random_split/processed/funct_train.tsv', sep='\t', index_col=0)

results = shuffle_and_cv(
        M=X_train.values,
        outdir='../results/functions/random_split/biCV',
        ranks=TOPICS,
        maxiter=1000,
        nruns=10,
        alpha=0,
        h=3
    )